## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <iostream>
#include <vector>
#include <queue>
#include <algorithm>
#include <string>
using namespace std;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    
    int k;
    cin >> k;
    int n = 1 << k;
    int a, b;
    cin >> a >> b;
    vector<int> p(n), pos(n);
    for (int i = 0; i < n; ++i) {
        cin >> p[i];
        pos[p[i]] = i;
    }
    
    // 检查是否已经有序
    bool sorted = true;
    for (int i = 0; i < n; ++i) {
        if (p[i] != i) { sorted = false; break; }
    }
    if (sorted) {
        cout << 0 << '\n';
        return 0;
    }
    
    // ---------- BFS 预处理：从 {a,b} 和 {b,a} 出发 ----------
    vector<vector<int>> pre_u(n, vector<int>(n, -1));
    vector<vector<int>> pre_v(n, vector<int>(n, -1));
    vector<vector<int>> op_type(n, vector<int>(n, -1)); // 0: add1, 1: xor1
    
    queue<pair<int,int>> q;
    pre_u[a][b] = -2; // 标记起点
    pre_u[b][a] = -2;
    q.push({a, b});
    q.push({b, a});
    
    while (!q.empty()) {
        auto [u, v] = q.front(); q.pop();
        // 加1
        int nu = (u + 1) % n, nv = (v + 1) % n;
        if (pre_u[nu][nv] == -1) {
            pre_u[nu][nv] = u;
            pre_v[nu][nv] = v;
            op_type[nu][nv] = 0;
            q.push({nu, nv});
        }
        // 异或1
        nu = u ^ 1; nv = v ^ 1;
        if (pre_u[nu][nv] == -1) {
            pre_u[nu][nv] = u;
            pre_v[nu][nv] = v;
            op_type[nu][nv] = 1;
            q.push({nu, nv});
        }
    }
    
    vector<string> ans;
    
    // 获取从目标状态到 (x,y) 的正向操作序列（加1/异或1）
    auto get_path = [&](int x, int y) {
        vector<pair<int,int>> path; // (type, param)
        int u = x, v = y;
        while (!((u == a && v == b) || (u == b && v == a))) {
            int op = op_type[u][v];
            if (op == 0) path.push_back({2, 1});      // 加1
            else path.push_back({1, 1});              // 异或1
            int pu = pre_u[u][v];
            int pv = pre_v[u][v];
            u = pu; v = pv;
        }
        reverse(path.begin(), path.end());
        return path;
    };
    
    // 排序过程
    for (int i = 0; i < n; ++i) {
        if (p[i] == i) continue;
        int j = pos[i];             // 数字 i 所在位置
        int x = p[i], y = i;
        if (x == y) continue;
        
        // 如果已经是目标对，直接交换
        if ((x == a && y == b) || (x == b && y == a)) {
            ans.push_back("0");
            swap(p[i], p[j]);
            pos[p[i]] = i;
            pos[p[j]] = j;
            continue;
        }
        
        auto path = get_path(x, y);   // 从 {a,b} 到 (x,y) 的正向操作序列
        
        // σ 序列：path 的逆，且每个操作取逆
        for (int k = path.size() - 1; k >= 0; --k) {
            auto [t, param] = path[k];
            if (t == 2) ans.push_back("2 " + to_string(n - 1)); // 加法的逆
            else        ans.push_back("1 1");                   // 异或的逆
        }
        // 交换魔法
        ans.push_back("0");
        // σ^{-1} 序列：即 path 本身
        for (auto [t, param] : path) {
            if (t == 2) ans.push_back("2 1");
            else        ans.push_back("1 1");
        }
        
        // 更新排列
        swap(p[i], p[j]);
        pos[p[i]] = i;
        pos[p[j]] = j;
    }
    
    cout << ans.size() << '\n';
    for (const auto& s : ans) cout << s << '\n';
    
    return 0;
}


## B 长跑

In [ ]:
import sys

def main():
    data = map(int, sys.stdin.read().split())
    ptr = 0
    total = len(data)
    while ptr < total:
        N = data[ptr]
        L = data[ptr+1]
        Maxn = data[ptr+2]
        S = data[ptr+3]
        ptr += 4
        
        station_dict = dict()
        for _ in range(N):
            Pi = data[ptr]
            Ci = data[ptr+1]
            ptr += 2
            if Pi >= L:
                continue
            if Pi not in station_dict or Ci < station_dict[Pi]:
                station_dict[Pi] = Ci
        
        if L == 0 or Maxn >= L:
            print "Yes"
            continue
        
        dp = [Maxn] * (S + 1)
        stations = station_dict.items()
        for pos, cost in stations:
            if cost > S:
                continue
            for k in range(S, cost - 1, -1):
                if dp[k - cost] >= pos:
                    new_reach = pos + Maxn
                    if new_reach > dp[k]:
                        dp[k] = new_reach
        
        if dp[S] >= L:
            print "Yes"
        else:
            print "No"

if __name__ == "__main__":
    main()


## C 最长回文

In [ ]:
#include <iostream>
#include <string>
#include <algorithm>
#include <vector>
using namespace std;

int manacher(const string& s) {
    string t = "#";
    for (char c : s) {
        t += c;
        t += '#';
    }
    int len = t.size();
    vector<int> p(len, 0);
    int C = 0, R = 0, max_len = 0;
    for (int i = 0; i < len; ++i) {
        int mirror = 2 * C - i;
        if (i < R) p[i] = min(R - i, p[mirror]);
        int l = i - p[i] - 1, r = i + p[i] + 1;
        while (l >= 0 && r < len && t[l] == t[r]) {
            p[i]++;
            l--;
            r++;
        }
        if (i + p[i] > R) {
            C = i;
            R = i + p[i];
        }
        max_len = max(max_len, p[i]);
    }
    return max_len;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;
    string A, B;
    cin >> A >> B;

    int ans = 0;
    for (int k = 0; k <= n; ++k) {
        string s = A.substr(0, k) + B.substr(k);
        ans = max(ans, manacher(s));
    }

    cout << ans << endl;
    return 0;
}


## D 优惠券

In [ ]:
#include <iostream>
#include <cstring>
#include <string>
using namespace std;

const int MAXV = 100005;
bool have[MAXV]; // have[x] == true 表示当前持有优惠券 x

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    
    int m;
    while (cin >> m) {
        memset(have, 0, sizeof(have));
        int wild = 0;   // 可以任意解释的 '?' 个数
        int ans = -1;   // 最早出错的行号
        
        for (int i = 1; i <= m; ++i) {
            string op;
            cin >> op;
            // 兼容半角 '?' 和全角 '？'
            if (op == "?" || op == "？") {
                if (ans == -1) ++wild;
            } else {
                int x;
                cin >> x;
                if (ans == -1) {
                    if (op == "I") {
                        if (!have[x]) {
                            have[x] = true;
                        } else { // have[x] == true，重复购买
                            if (wild > 0) {
                                --wild;       // 消耗一个 '?' 当作一次使用
                                // have[x] 仍为 true（相当于用 ? 释放后又马上买了）
                            } else {
                                ans = i;
                            }
                        }
                    } else { // op == "O"
                        if (have[x]) {
                            have[x] = false;
                        } else { // 没有券可花
                            if (wild > 0) {
                                --wild;       // 消耗一个 '?' 当作购买
                                // 不改变 have[x]（相当于 ? 购买后立即使用）
                            } else {
                                ans = i;
                            }
                        }
                    }
                }
            }
        }
        cout << ans << '\n';
    }
    return 0;
}


## E 任意点

In [ ]:
def find(u, parent):
    if parent[u] != u:
        parent[u] = find(parent[u], parent)
    return parent[u]

def union(u, v, parent):
    u_root = find(u, parent)
    v_root = find(v, parent)
    if u_root != v_root:
        parent[v_root] = u_root

def main():
    import sys
    data = map(int, sys.stdin.read().split())
    ptr = 0
    n = int(data[ptr])
    ptr += 1
    max_id = 2000
    parent = range(max_id + 1)
    seen = set()
    for _ in range(n):
        x = data[ptr]
        y = data[ptr + 1]
        ptr += 2
        y_id = y + 1000
        union(x, y_id, parent)
        seen.add(x)
        seen.add(y_id)
    roots = set()
    for num in seen:
        roots.add(find(num, parent))
    print len(roots) - 1

if __name__ == "__main__":
    main()


## F 通配符匹配

In [ ]:
# -*- coding: utf-8 -*-
import sys

def kmp_search(text, pattern, start_pos, end_limit):
    """
    在 text[start_pos : end_limit + len(pattern)] 范围内查找 pattern 第一次出现的位置。
    pattern 中允许 '?' 通配符，匹配任意单个字符。
    返回匹配的起始下标，若未找到返回 -1。
    """
    if not pattern:
        return start_pos

    n = len(text)
    m = len(pattern)

    # 构建 next 数组（支持 '?' 通配符）
    next_arr = [0] * m
    j = 0
    for i in range(1, m):
        while j > 0 and pattern[i] != pattern[j] and pattern[i] != '?' and pattern[j] != '?':
            j = next_arr[j-1]
        if pattern[i] == pattern[j] or pattern[i] == '?' or pattern[j] == '?':
            j += 1
            next_arr[i] = j
        # else next_arr[i] 保持 0

    # KMP 主匹配过程
    j = 0
    i = start_pos
    while i < n:
        # 关键修正：检查当前可能的匹配起始位置是否已超出 end_limit
        if i - j > end_limit:
            return -1

        if j < m and (pattern[j] == '?' or text[i] == pattern[j]):
            i += 1
            j += 1
        elif j == m:
            start = i - m
            if start <= end_limit:
                return start
            else:
                return -1
        else:
            if j > 0:
                j = next_arr[j-1]
            else:
                i += 1

    # 处理文本末尾恰好匹配完成的情况
    if j == m:
        start = i - m
        if start <= end_limit:
            return start
    return -1


def match_segment(text, segment, start_pos):
    """检查 text 从 start_pos 开始是否与 segment 完全匹配（支持 '?'）"""
    if start_pos + len(segment) > len(text):
        return False
    for i in range(len(segment)):
        if segment[i] != '?' and segment[i] != text[start_pos + i]:
            return False
    return True


def match_segment_at_end(text, segment):
    """检查 text 结尾是否与 segment 匹配"""
    if len(segment) > len(text):
        return False
    start = len(text) - len(segment)
    return match_segment(text, segment, start)


def match_pattern(filename, pattern):
    """判断文件名是否匹配带通配符的模式串"""
    if not pattern:
        return not filename

    if all(ch == '*' for ch in pattern):
        return True

    parts = pattern.split('*')
    first_part = parts[0]
    last_part = parts[-1]
    middle_parts = parts[1:-1] if len(parts) > 2 else []

    # 1. 处理开头（如果模式串不以 '*' 开头）
    if pattern[0] != '*':
        if not match_segment(filename, first_part, 0):
            return False
        pos = len(first_part)
    else:
        pos = 0

    # 2. 处理结尾（如果模式串不以 '*' 结尾）
    if pattern[-1] != '*':
        if not match_segment_at_end(filename, last_part):
            return False
        end_limit = len(filename) - len(last_part)
    else:
        end_limit = len(filename)

    # 3. 贪心匹配中间片段（使用修正后的 KMP）
    for part in middle_parts:
        if not part:  # 忽略空片段（连续的 '*' 产生）
            continue
        found_pos = kmp_search(filename, part, pos, end_limit)
        if found_pos == -1:
            return False
        pos = found_pos + len(part)

    # 确保中间部分没有超出结尾限制（无论开头是否有 '*'，只要 pos 超过 end_limit 即失败）
    if pos > end_limit:
        return False

    return True


def main():
    data = sys.stdin.read().splitlines()
    if not data:
        return
    pattern = data[0].strip()
    n = int(data[1].strip())
    for i in range(2, 2 + n):
        filename = data[i].strip()
        if match_pattern(filename, pattern):
            print "YES"
        else:
            print "NO"


if __name__ == "__main__":
    main()


## G 汉诺塔

In [ ]:
import sys

def simulate_small(n_small, ops):
    char_map = {'A':0, 'B':1, 'C':2}
    op_list = []
    for s in ops:
        src = char_map[s[0]]
        dst = char_map[s[1]]
        op_list.append( (src, dst) )
    
    pegs = [ [], [], [] ]
    for i in range(n_small, 0, -1):
        pegs[0].append(i)
    
    last_disk = None
    step = 0
    
    while True:
        if len(pegs[1]) == n_small or len(pegs[2]) == n_small:
            break
        found = False
        for src, dst in op_list:
            if not pegs[src]:
                continue
            d = pegs[src][-1]
            if d == last_disk:
                continue
            if pegs[dst] and pegs[dst][-1] < d:
                continue
            pegs[src].pop()
            pegs[dst].append(d)
            step += 1
            last_disk = d
            found = True
            break
        if not found:
            break
    return step

def main():
    data = sys.stdin.read().split()
    ptr = 0
    n = int(data[ptr])
    ptr += 1
    ops = data[ptr:ptr+6]
    
    if n == 0:
        print 0
        return
    if n == 1:
        print 1
        return
    
    f1 = simulate_small(1, ops)
    f2 = simulate_small(2, ops)
    f3 = simulate_small(3, ops)
    
    a = (f3 - f2) // (f2 - f1)
    b = f2 - a * f1
    
    prev = f1
    for k in range(2, n + 1):
        curr = a * prev + b
        prev = curr
    print prev

if __name__ == "__main__":
    main()


## H 马步距离

In [ ]:
def main():
    import sys
    data = map(int, sys.stdin.read().split())
    xp = data[0]
    yp = data[1]
    xs = data[2]
    ys = data[3]
    
    dx = abs(xs - xp)
    dy = abs(ys - yp)
    
    if dx < dy:
        dx, dy = dy, dx
    
    if dx == 0 and dy == 0:
        print 0
        return
    if dx == 1 and dy == 0:
        print 3
        return
    if dx == 2 and dy == 2:
        print 4
        return
    
    k0 = max((dx + 1) // 2, (dx + dy + 2) // 3)
    if (k0 % 2) != ((dx + dy) % 2):
        k0 += 1
    print k0

if __name__ == "__main__":
    main()


## I 直方图最大矩形

In [ ]:
#coding:utf-8
#
# 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
#
# 
# @param heights int整型一维数组 
# @return int整型
#
class Solution:
    def largestRectangleArea(self , heights ):
        # 空数组直接返回0
        if not heights:
            return 0
        # 首尾补0，统一处理边界情况
        h = [0] + heights + [0]
        stack = []
        max_area = 0
        # 单调递增栈遍历
        for i in range(len(h)):
            # 遇到更小的高度，弹出栈顶计算面积
            while stack and h[i] < h[stack[-1]]:
                top_idx = stack.pop()
                current_height = h[top_idx]
                # 宽度 = 右边界索引 - 左边界索引 - 1
                current_width = i - stack[-1] - 1
                current_area = current_height * current_width
                if current_area > max_area:
                    max_area = current_area
            stack.append(i)
        return max_area


## J 消防局的设立

In [ ]:
import sys
INF = 10**9

def main():
    data = map(int, sys.stdin.read().split())
    ptr = 0
    n = int(data[ptr])
    ptr += 1
    if n == 1:
        print(1)
        return
    g = [[] for _ in range(n + 1)]
    for i in range(2, n + 1):
        a = int(data[ptr])
        ptr += 1
        g[i].append(a)
        g[a].append(i)
    
    dp0 = [0]*(n+1)
    dp1 = [0]*(n+1)
    dp2 = [0]*(n+1)
    parent = [-1]*(n+1)
    stack = [(1, False)]
    
    while stack:
        u, visited = stack.pop()
        if not visited:
            stack.append((u, True))
            for v in g[u]:
                if v != parent[u]:
                    parent[v] = u
                    stack.append((v, False))
        else:
            is_leaf = True
            min_diff = INF
            dp0_u = 1
            dp1_u = 0
            dp2_u = 0
            for v in g[u]:
                if v == parent[u]:
                    continue
                is_leaf = False
                v0, v1, v2 = dp0[v], dp1[v], dp2[v]
                dp0_u += min(v0, v1, v2)
                dp1_u += v2
                min_v = min(v0, v2)
                dp2_u += min_v
                diff = v0 - min_v
                if diff < min_diff:
                    min_diff = diff
            if is_leaf:
                dp0[u] = 1
                dp1[u] = INF
                dp2[u] = 0
            else:
                dp0[u] = dp0_u
                dp1[u] = dp1_u
                dp2[u] = dp2_u + min_diff
    
    ans = min(dp0[1], dp2[1])
    print(ans)

if __name__ == "__main__":
    main()
